# nonlocal
- nonlocal 是 Python 中<font color='red'>用于闭包（嵌套函数）修改外层（非全局）变量的关键字</font>。
- 在 VPP 调度代码中大量使用，是实现状态共享的核心机制。

## 核心问题：为什么需要 nonlocal？

### 示例：不用 nonlocal 的问题
- 问题：inner() 中的 count = 1 创建了一个新的局部变量，而不是修改外层的 count

In [ ]:
def outer():
    count = 0  # 外层函数的局部变量
    
    def inner():
        count = 1  # ← 这里创建了一个新的局部变量！
        print(f"inner: {count}")  # 输出 1
    
    inner()
    print(f"outer: {count}")  # 输出 0（没有被修改！）

outer()

### nonlocal 的解决方案

In [ ]:
def outer():
    count = 0  # 外层函数的局部变量
    
    def inner():
        nonlocal count  # ← 声明：我要修改外层的 count
        count = 1       # ← 现在修改的是外层的变量！
        print(f"inner: {count}")  # 输出 1
    
    inner()
    print(f"outer: {count}")  # 输出 1（被成功修改！）

outer()

## 三个关键字对比

| 关键字        | 作用域           | 修改目标     | VPP 中使用场景    |
| ---------- | ------------- | -------- | ------------ |
| `global`   | 模块级           | 全局变量     | 极少使用（避免全局状态） |
| `nonlocal` | **外层函数的局部变量** | 闭包中的共享状态 | **大量使用**     |
| 无（默认）      | 当前函数          | 创建新的局部变量 | 普通局部变量       |


## 在 VPP 代码中的实际应用
### 场景：Steady 1F1B 的回调函数


In [ ]:
def run_steady_1f1b():
    # ========== 外层函数定义的共享状态 ==========
    send_next_wait_handle = None      # 发送句柄
    fwd_recv_buffer = [None] * 2      # 接收缓冲区
    fwd_wait_handles = None           # P2P 句柄字典
    recv_prev_wait_handles = []       # 接收句柄队列
    
    # ========== 内层回调函数（闭包）==========
    
    def pp_pre_forward(vp_stage=None):
        nonlocal recv_prev_wait_handles  # ← 修改外层的队列
        ...
        recv_prev_wait_handle = recv_prev_wait_handles.pop(0)  # 消费队列
        ...
    
    def pp_post_forward(output_tensor, vp_stage=None):
        nonlocal send_next_wait_handle   # ← 修改外层的发送句柄
        nonlocal fwd_recv_buffer         # ← 修改外层的缓冲区
        nonlocal fwd_wait_handles        # ← 修改外层的句柄
        nonlocal recv_prev_wait_handles  # ← 修改外层的队列
        
        ...
        send_next_wait_handle = fwd_wait_handles.pop("send_next")  # 更新发送句柄
        recv_prev_wait_handles.append(...)  # 填充队列
        ...
    
    def pp_pre_backward(vp_stage=None):
        nonlocal recv_next_wait_handles  # ← 修改外层的反向接收队列
        ...
    
    def pp_post_backward(input_tensor_grad, vp_stage=None):
        nonlocal send_prev_wait_handle   # ← 修改外层的发送句柄
        nonlocal bwd_wait_handles        # ← 修改外层的反向句柄
        ...
    
    # ========== 主循环使用回调 ==========
    for k in range(num_microbatches_remaining):
        output_tensor, input_grad = forward_backward_helper_wrapper(
            f_virtual_microbatch_id=forward_k,
            b_virtual_microbatch_id=backward_k,
            pre_forward=pp_pre_forward,    # 传入闭包
            post_forward=pp_post_forward,  # 传入闭包
            pre_backward=pp_pre_backward,  # 传入闭包
            post_backward=pp_post_backward, # 传入闭包
        )

### 为什么不用其他方案？
#### 方案对比

| 方案               | 代码示例                             | 问题             |
| ---------------- | -------------------------------- | -------------- |
| **全局变量**         | `global send_next_wait_handle`   | 污染全局命名空间，多实例冲突 |
| **类封装**          | `self.send_next_wait_handle`     | 需要定义类，代码更冗长    |
| **返回值传递**        | `return send_next_wait_handle`   | 每次都要返回和重新赋值，繁琐 |
| **`nonlocal` ✅** | `nonlocal send_next_wait_handle` | <font color='red'>简洁、安全、符合闭包语义</font>   |


In [ ]:
外层函数 run_steady_1f1b() 的局部变量（共享状态池）:
├─ send_next_wait_handle = None
├─ fwd_recv_buffer = [None, None]
├─ fwd_wait_handles = None
├─ recv_prev_wait_handles = []
└─ recv_next_wait_handles = []
         ↑
         │ nonlocal 声明
    ┌────┼────┐
    ↓    ↓    ↓
pp_pre_forward  pp_post_forward  pp_pre_backward  pp_post_backward
（闭包函数）    （闭包函数）      （闭包函数）      （闭包函数）

执行时序:
1. pp_post_forward(): 
   nonlocal send_next_wait_handle → 创建句柄 → 赋值给外层变量
   
2. 下次 pp_post_forward():
   nonlocal send_next_wait_handle → 读取上次赋值 → 等待完成
   
3. pp_pre_forward():
   nonlocal recv_prev_wait_handles → 读取队列 → pop 消费

## 多层嵌套时，nonlocal 找到最近的定义

In [ ]:
def a():
    x = 1
    def b():
        x = 2  # b 的局部变量
        def c():
            nonlocal x  # ← 找到 b 的 x，不是 a 的 x
            x = 3
        c()
        print(f"b.x = {x}")  # 3
    b()
    print(f"a.x = {x}")  # 1（未被修改）

a()

## 一句话总结
- nonlocal 是 Python 闭包中的"状态共享契约"：
  - 让嵌套函数能够读写外层函数的局部变量，而不是创建隔离的副本。
  - <font color='red'>在 VPP 调度中，它让 pp_pre_forward/post_forward 等回调函数能够协同操作同一套通信句柄和缓冲区，实现跨迭代的异步状态管理</font>
  - 没有 nonlocal，每次回调都会丢失上次的通信状态，重叠通信机制将彻底失效。